[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/E.%20Strategic%20Design%20%26%20Long-Term%20Investment%20%28Shaping%20the%20Future%29/45.%20The%20Terminal%20Digital%20Twin%20Scoping%20Problem/P45-Tier-1_executed.ipynb)

---


# 45. The Terminal Digital Twin Scoping Problem
## Tier 1: The Pen-and-Paper Method (Mathematical Formulation)

### Goal
Choose which terminal components to include in a digital twin and which sensor locations to use. The model must satisfy budget, resource, complexity, minimum-value, and coverage requirements. Among feasible choices, it maximizes a value-to-cost score adjusted by a complexity penalty.

### Inputs, outputs, and scope
The inputs are component values, costs, complexity scores, and resource requirements; sensor installation costs; a component–sensor coverage matrix; and the limits and penalty parameter. The output is an optimal feasible selection of components and sensors, together with its objective value and constraint totals. Costs are expressed in thousands of dollars, resources in person-hours, and component value and complexity are scores in the units defined by the example.

The model treats each component and sensor location as a binary choice. It assumes that component values, costs, resource requirements, complexity scores, and coverage are known and fixed. Sensor locations have installation costs and coverage relationships, but no separate resource or complexity requirements. No additional component interdependencies are specified in the data, so the model does not impose interdependency constraints. Uncertainty, time-varying operations, and sensor capacity are outside this example's scope.

### Example instance
The Mediterranean Container Terminal (MCT) example contains three components—Quay Crane, Yard Block, and Gate System—and four potential sensor locations. Its budget limit is $250,000 and its resource limit is 80 person-hours. Component values are `[85, 65, 75]`, component costs are `[120, 80, 95]` thousand dollars, complexity scores are `[3, 2, 4]`, and resource requirements are `[40, 25, 35]` person-hours. The code cell defining the instance supplies sensor costs and coverage relationships.

### Method and learning objectives
The solution method is exact exhaustive enumeration for this small instance. It examines every nonempty component subset, finds a minimum-cost sensor set that covers that subset, checks feasibility, and retains a feasible solution with the largest objective value. This is a transparent exact method for small instances; it does not call a mixed-integer programming solver. The sensitivity analysis repeats the same enumeration while varying selected limits or the complexity-penalty parameter.

As you work through the notebook, trace how the binary decisions become a mathematical model, how coverage links component choices to sensor choices, and how exhaustive enumeration evaluates each candidate. The results show both the selected scope and the trade-offs represented by the model.

## Mathematical formulation

The model makes two linked decisions: whether to include each component and whether to install each sensor. Let `x_i` be 1 when component `i` is selected and 0 otherwise; let `y_l` be 1 when sensor location `l` is selected and 0 otherwise. The coverage parameter `a_{il}` is 1 if sensor `l` can monitor component `i`, and 0 otherwise.

For any selected components, the objective rewards operational value relative to total implementation cost and subtracts a penalty for complexity:

$$
\max_{x,y}\quad
\frac{\sum_{i=1}^{n} v_i x_i}
{\sum_{i=1}^{n} c_i x_i + \sum_{l=1}^{m} s_l y_l}
- \lambda \sum_{i=1}^{n} d_i x_i
$$

subject to

$$
\begin{aligned}
\sum_{i=1}^{n} c_i x_i + \sum_{l=1}^{m} s_l y_l &\le B, \\
\sum_{i=1}^{n} r_i x_i &\le R, \\
\sum_{i=1}^{n} d_i x_i &\le D, \\
\sum_{i=1}^{n} v_i x_i &\ge V_{\min}, \\
\sum_{l=1}^{m} a_{il} y_l &\ge x_i && i=1,\ldots,n, \\
x_i,y_l &\in \{0,1\} && i=1,\ldots,n,\quad l=1,\ldots,m.
\end{aligned}
$$

Here, `v_i` is component operational value, `c_i` its implementation cost, `s_l` the installation cost of sensor `l`, `d_i` the component complexity score, and `r_i` its resource requirement. `B`, `R`, and `D` are the budget, resource, and complexity limits; `V_min` is the minimum required value; and `lambda` is the complexity-penalty parameter. The denominator is total implementation cost, including both components and sensors. In the code, these quantities are stored on `Component` and `SensorLocation` objects or passed to `DigitalTwinScopingSolver`.

The coverage inequality requires at least one selected sensor to cover every selected component. Positive sensor costs mean that an optimal solution has no reason to install an unnecessary sensor. The minimum-value requirement excludes the empty component selection in this instance, so every evaluated candidate has positive total cost and a defined ratio.

### How exhaustive enumeration implements the model

The solver enumerates each nonempty component subset. For a fixed subset, its value, component cost, resource use, and complexity are fixed. The solver enumerates sensor subsets and retains a minimum-cost subset that covers all selected components. This is sufficient for the stated objective and constraints: for a fixed component subset, a more expensive covering sensor set cannot improve the value-to-cost ratio, and sensors do not affect the other totals. It then evaluates the resulting candidate against all constraints.

For `n` components and `m` sensor locations, there are `2^n - 1` nonempty component subsets. In the worst case, searching sensor subsets for each component subset requires examining up to `2^m - 1` nonempty sensor subsets, so the method grows exponentially and is intended for small instances. If several sensor subsets have the same minimum cost, the enumeration retains the first one encountered. If several feasible component selections have equal objective values, the solver retains the first one encountered; floating-point comparisons use a tolerance when deciding whether objective values are effectively equal.

In [ ]:
# Import the packages used for exact enumeration and visualization
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from math import isclose

# A consistent plotting style makes the solution and sensitivity plots easier to compare.
plt.style.use("default")
sns.set_palette("husl")


In [ ]:
# Each component stores the quantities that contribute to the selection model.
class Component:
    """A terminal component that may be included in the digital twin."""

    def __init__(self, id, name, value, cost, complexity, resources):
        self.id = id
        self.name = name
        self.value = value
        self.cost = cost  # Implementation cost in thousands of dollars.
        self.complexity = complexity
        self.resources = resources  # Requirement in person-hours.

    def __repr__(self):
        return (
            f"Component({self.id}: {self.name}, V={self.value}, C={self.cost}, "
            f"D={self.complexity}, R={self.resources})"
        )


# A sensor location contributes installation cost and coverage capability.
class SensorLocation:
    """A potential sensor location for monitoring terminal components."""

    def __init__(self, id, name, cost):
        self.id = id
        self.name = name
        self.cost = cost  # Installation cost in thousands of dollars.

    def __repr__(self):
        return f"Sensor({self.id}: {self.name}, Cost=${self.cost}k)"


print("Component and sensor data structures are ready.")


In [ ]:
# Initialize the MCT example with the stated component and sensor data.
components = [
    Component(1, "Quay Crane", value=85, cost=120, complexity=3, resources=40),
    Component(2, "Yard Block", value=65, cost=80, complexity=2, resources=25),
    Component(3, "Gate System", value=75, cost=95, complexity=4, resources=35),
]

sensor_locations = [
    SensorLocation(1, "Quay Area Sensor", cost=30),
    SensorLocation(2, "Yard Block Sensor", cost=25),
    SensorLocation(3, "Gate System Sensor", cost=35),
    SensorLocation(4, "Perimeter Sensor", cost=40),
]

# Row i and column l describe whether sensor l can monitor component i.
coverage_matrix = np.array(
    [
        [1, 1, 0, 0],  # Quay Crane is covered by sensors 1 or 2.
        [0, 1, 1, 0],  # Yard Block is covered by sensors 2 or 3.
        [0, 0, 1, 1],  # Gate System is covered by sensors 3 or 4.
    ]
)

budget_limit = 250  # Thousands of dollars.
resource_limit = 80  # Person-hours.
complexity_limit = 10
min_value_required = 100
lambda_penalty = 0.01

print("Problem instance initialized:")
print(f"Components: {len(components)}")
print(f"Sensor locations: {len(sensor_locations)}")
print(f"Budget limit: ${budget_limit}k")
print(f"Resource limit: {resource_limit} person-hours")
print(f"Complexity limit: {complexity_limit}")
print("\nComponents:")
for component in components:
    print(f"  {component}")
print("\nSensors:")
for sensor in sensor_locations:
    print(f"  {sensor}")


In [ ]:
class DigitalTwinScopingSolver:
    """Evaluate the scoping model by exact exhaustive enumeration."""

    def __init__(
        self,
        components,
        sensors,
        coverage_matrix,
        budget_limit,
        resource_limit,
        complexity_limit,
        min_value_required,
        lambda_penalty=0.01,
    ):
        self.components = components
        self.sensors = sensors
        self.coverage_matrix = np.asarray(coverage_matrix)
        self.budget_limit = budget_limit
        self.resource_limit = resource_limit
        self.complexity_limit = complexity_limit
        self.min_value_required = min_value_required
        self.lambda_penalty = lambda_penalty
        self.n_components = len(components)
        self.n_sensors = len(sensors)

        if self.coverage_matrix.shape != (self.n_components, self.n_sensors):
            raise ValueError("The coverage matrix must have one row per component and one column per sensor.")

    def evaluate_solution(self, selected_components, selected_sensors):
        """Calculate candidate totals, objective value, and constraint feasibility."""
        total_value = sum(self.components[i].value for i in selected_components)
        total_cost = sum(self.components[i].cost for i in selected_components)
        total_cost += sum(self.sensors[j].cost for j in selected_sensors)
        total_resources = sum(
            self.components[i].resources for i in selected_components
        )
        total_complexity = sum(
            self.components[i].complexity for i in selected_components
        )

        budget_feasible = total_cost <= self.budget_limit
        resource_feasible = total_resources <= self.resource_limit
        complexity_feasible = total_complexity <= self.complexity_limit
        value_feasible = total_value >= self.min_value_required

        # Every selected component must have at least one selected covering sensor.
        coverage_feasible = all(
            any(self.coverage_matrix[i][j] for j in selected_sensors)
            for i in selected_components
        )

        feasible = all(
            [
                budget_feasible,
                resource_feasible,
                complexity_feasible,
                value_feasible,
                coverage_feasible,
            ]
        )

        # The empty selection has no defined value-to-cost ratio.
        if total_cost > 0:
            objective = (
                total_value / total_cost
                - self.lambda_penalty * total_complexity
            )
        else:
            objective = -float("inf")

        return {
            "objective": objective,
            "total_value": total_value,
            "total_cost": total_cost,
            "total_resources": total_resources,
            "total_complexity": total_complexity,
            "feasible": feasible,
            "budget_feasible": budget_feasible,
            "resource_feasible": resource_feasible,
            "complexity_feasible": complexity_feasible,
            "value_feasible": value_feasible,
            "coverage_feasible": coverage_feasible,
        }

    def find_minimal_sensors(self, selected_components):
        """Return a minimum-cost sensor subset covering the selected components."""
        minimal_sensors = None
        minimal_cost = float("inf")

        # Sensor subsets are enumerated because each location has a binary choice.
        for subset_size in range(1, self.n_sensors + 1):
            for sensor_combo in combinations(range(self.n_sensors), subset_size):
                covers_all = all(
                    any(self.coverage_matrix[i][j] for j in sensor_combo)
                    for i in selected_components
                )
                if not covers_all:
                    continue

                sensor_cost = sum(self.sensors[j].cost for j in sensor_combo)
                if sensor_cost < minimal_cost:
                    minimal_cost = sensor_cost
                    minimal_sensors = list(sensor_combo)

        return minimal_sensors if minimal_sensors is not None else []

    def solve_exhaustive(self):
        """Enumerate nonempty component selections and retain the best feasible one."""
        best_solution = None
        best_objective = -float("inf")
        all_solutions = []

        for subset_size in range(1, self.n_components + 1):
            for component_combo in combinations(
                range(self.n_components), subset_size
            ):
                selected_components = list(component_combo)
                selected_sensors = self.find_minimal_sensors(selected_components)
                solution = self.evaluate_solution(
                    selected_components, selected_sensors
                )
                solution["selected_components"] = selected_components
                solution["selected_sensors"] = selected_sensors
                all_solutions.append(solution)

                if not solution["feasible"]:
                    continue

                # Retaining the first candidate within tolerance makes tie handling
                # stable with respect to the enumeration order.
                if solution["objective"] > best_objective and not isclose(
                    solution["objective"],
                    best_objective,
                    rel_tol=1e-12,
                    abs_tol=1e-12,
                ):
                    best_objective = solution["objective"]
                    best_solution = solution

        return best_solution, all_solutions


print("Exact exhaustive solver is ready.")


In [ ]:
solver = DigitalTwinScopingSolver(
    components=components,
    sensors=sensor_locations,
    coverage_matrix=coverage_matrix,
    budget_limit=budget_limit,
    resource_limit=resource_limit,
    complexity_limit=complexity_limit,
    min_value_required=min_value_required,
    lambda_penalty=lambda_penalty,
)

best_solution, all_solutions = solver.solve_exhaustive()
feasible_solutions = [solution for solution in all_solutions if solution["feasible"]]

print("=== MATHEMATICAL FORMULATION RESULTS ===")
print(f"\nOptimal solution found: {best_solution is not None}")

if best_solution is not None:
    print("\n--- BEST FEASIBLE SOLUTION ---")
    print("Selected components:")
    for i in best_solution["selected_components"]:
        component = components[i]
        print(
            f"  - {component.name}: Value={component.value}, "
            f"Cost=${component.cost}k, Complexity={component.complexity}, "
            f"Resources={component.resources} person-hours"
        )

    print("\nSelected sensors:")
    for j in best_solution["selected_sensors"]:
        sensor = sensor_locations[j]
        print(f"  - {sensor.name}: Cost=${sensor.cost}k")

    print("\nSolution summary:")
    print(f"  Total operational value: {best_solution['total_value']}")
    print(f"  Total implementation cost: ${best_solution['total_cost']}k")
    print(f"  Total resources used: {best_solution['total_resources']} person-hours")
    print(f"  Total system complexity: {best_solution['total_complexity']}")
    print(
        "  Value-to-cost ratio: "
        f"{best_solution['total_value'] / best_solution['total_cost']:.3f}"
    )
    print(f"  Objective function value: {best_solution['objective']:.4f}")

    print("\nConstraint checks:")
    print(
        f"  Budget: ${best_solution['total_cost']}k <= ${budget_limit}k — "
        f"{'satisfied' if best_solution['budget_feasible'] else 'violated'}"
    )
    print(
        f"  Resources: {best_solution['total_resources']} <= {resource_limit} "
        f"person-hours — {'satisfied' if best_solution['resource_feasible'] else 'violated'}"
    )
    print(
        f"  Complexity: {best_solution['total_complexity']} <= {complexity_limit} — "
        f"{'satisfied' if best_solution['complexity_feasible'] else 'violated'}"
    )
    print(
        f"  Minimum value: {best_solution['total_value']} >= "
        f"{min_value_required} — "
        f"{'satisfied' if best_solution['value_feasible'] else 'violated'}"
    )
    print(
        "  Coverage: "
        f"{'satisfied' if best_solution['coverage_feasible'] else 'violated'}"
    )

    # These checks connect the displayed plan to its evaluated totals and constraints.
    assert best_solution["feasible"]
    assert best_solution["total_cost"] == sum(
        components[i].cost for i in best_solution["selected_components"]
    ) + sum(sensor_locations[j].cost for j in best_solution["selected_sensors"])
    assert best_solution["total_value"] == sum(
        components[i].value for i in best_solution["selected_components"]
    )
    assert all(
        any(coverage_matrix[i][j] for j in best_solution["selected_sensors"])
        for i in best_solution["selected_components"]
    )
else:
    print("No feasible solution was found within the stated limits.")

print(f"\nComponent selections evaluated: {len(all_solutions)}")
print(f"Feasible component selections: {len(feasible_solutions)}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
fig.suptitle(
    "Terminal Digital Twin Scoping: Exact Enumeration Results",
    fontsize=16,
    fontweight="bold",
)

# Candidate totals show how feasibility and the objective distinguish selections.
ax1 = axes[0, 0]
infeasible_solutions = [
    solution for solution in all_solutions if not solution["feasible"]
]
ax1.scatter(
    [solution["total_cost"] for solution in infeasible_solutions],
    [solution["total_value"] for solution in infeasible_solutions],
    c="red",
    alpha=0.5,
    s=30,
    label="Infeasible",
)
ax1.scatter(
    [solution["total_cost"] for solution in feasible_solutions],
    [solution["total_value"] for solution in feasible_solutions],
    c="green",
    alpha=0.7,
    s=50,
    label="Feasible",
)
if best_solution is not None:
    ax1.scatter(
        best_solution["total_cost"],
        best_solution["total_value"],
        c="gold",
        s=200,
        marker="*",
        edgecolors="black",
        linewidth=1.5,
        label="Best feasible",
    )
ax1.set_xlabel("Total implementation cost (thousands of dollars)")
ax1.set_ylabel("Total operational value (score)")
ax1.set_title("Candidate value and cost")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Bar color distinguishes selected components from those outside the chosen scope.
ax2 = axes[0, 1]
if best_solution is not None:
    selected_comp_indices = best_solution["selected_components"]
    comp_names = [component.name for component in components]
    comp_values = [component.value for component in components]
    bar_colors = [
        "green" if i in selected_comp_indices else "lightgray"
        for i in range(len(components))
    ]
    bars = ax2.bar(
        comp_names,
        comp_values,
        color=bar_colors,
        alpha=0.75,
        edgecolor="black",
    )
    ax2.set_ylabel("Operational value (score)")
    ax2.set_title("Component selection (green = selected)")
    ax2.tick_params(axis="x", rotation=35)
    for bar, value in zip(bars, comp_values):
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            str(value),
            ha="center",
            va="bottom",
            fontweight="bold",
        )
else:
    ax2.text(0.5, 0.5, "No feasible solution", ha="center", va="center", transform=ax2.transAxes)
    ax2.set_title("Component selection")

# Upper limits use used/limit; the minimum-value requirement uses value/minimum.
ax3 = axes[1, 0]
if best_solution is not None:
    constraint_names = ["Budget", "Resources", "Complexity", "Minimum value"]
    constraint_ratios = [
        best_solution["total_cost"] / budget_limit if budget_limit > 0 else np.nan,
        best_solution["total_resources"] / resource_limit if resource_limit > 0 else np.nan,
        best_solution["total_complexity"] / complexity_limit if complexity_limit > 0 else np.nan,
        best_solution["total_value"] / min_value_required
        if min_value_required > 0
        else np.nan,
    ]
    satisfied = [
        best_solution["budget_feasible"],
        best_solution["resource_feasible"],
        best_solution["complexity_feasible"],
        best_solution["value_feasible"],
    ]
    bar_colors = ["green" if is_satisfied else "red" for is_satisfied in satisfied]
    bars = ax3.bar(
        constraint_names,
        constraint_ratios,
        color=bar_colors,
        alpha=0.75,
        edgecolor="black",
    )
    ax3.axhline(y=1.0, color="black", linestyle="--", alpha=0.7, label="Constraint threshold")
    ax3.set_ylabel("Used / upper limit, or value / minimum")
    ax3.set_title("Constraint ratios (green = satisfied)")
    ax3.tick_params(axis="x", rotation=25)
    ax3.legend()
    for bar, ratio in zip(bars, constraint_ratios):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f"{ratio:.1%}",
            ha="center",
            va="bottom",
            fontsize=9,
        )
else:
    ax3.text(0.5, 0.5, "No feasible solution", ha="center", va="center", transform=ax3.transAxes)
    ax3.set_title("Constraint ratios")

# The matrix entries show coverage; outlined rows and columns identify selected items.
ax4 = axes[1, 1]
if best_solution is not None:
    image = ax4.imshow(coverage_matrix, cmap="Blues", vmin=0, vmax=1, alpha=0.8)
    ax4.set_xticks(range(len(sensor_locations)))
    ax4.set_yticks(range(len(components)))
    ax4.set_xticklabels([sensor.name for sensor in sensor_locations], rotation=35, ha="right")
    ax4.set_yticklabels([component.name for component in components])
    for i in range(len(components)):
        for j in range(len(sensor_locations)):
            ax4.text(
                j,
                i,
                str(coverage_matrix[i, j]),
                ha="center",
                va="center",
                color="black",
                fontweight="bold",
            )
    for i in best_solution["selected_components"]:
        ax4.add_patch(
            plt.Rectangle(
                (-0.5, i - 0.5),
                len(sensor_locations),
                1,
                fill=False,
                edgecolor="green",
                linewidth=2,
            )
        )
    for j in best_solution["selected_sensors"]:
        ax4.add_patch(
            plt.Rectangle(
                (j - 0.5, -0.5),
                1,
                len(components),
                fill=False,
                edgecolor="orange",
                linewidth=2,
            )
        )
    colorbar = fig.colorbar(image, ax=ax4, ticks=[0, 1], fraction=0.046, pad=0.04)
    colorbar.ax.set_yticklabels(["No coverage", "Coverage"])
    ax4.set_title("Coverage matrix: green rows = selected components; orange columns = selected sensors")
    ax4.set_xlabel("Sensor locations")
    ax4.set_ylabel("Components")
else:
    ax4.text(0.5, 0.5, "No feasible solution", ha="center", va="center", transform=ax4.transAxes)
    ax4.set_title("Coverage matrix")

plt.tight_layout()
plt.show()


In [ ]:
def sensitivity_analysis(
    components,
    sensor_locations,
    coverage_matrix,
    budget_limit,
    resource_limit,
    complexity_limit,
    min_value_required,
    lambda_penalty,
):
    """Repeat the exact enumeration while varying one model parameter at a time."""
    budget_range = [200, 250, 300, 350, 400]
    resource_range = [60, 80, 100, 120, 140]
    lambda_range = [0.005, 0.01, 0.02, 0.05, 0.1]

    results = {
        "budget_sensitivity": [],
        "resource_sensitivity": [],
        "lambda_sensitivity": [],
    }

    print("=== BUDGET SENSITIVITY ANALYSIS ===")
    for budget in budget_range:
        candidate_solver = DigitalTwinScopingSolver(
            components=components,
            sensors=sensor_locations,
            coverage_matrix=coverage_matrix,
            budget_limit=budget,
            resource_limit=resource_limit,
            complexity_limit=complexity_limit,
            min_value_required=min_value_required,
            lambda_penalty=lambda_penalty,
        )
        solution, _ = candidate_solver.solve_exhaustive()
        if solution is None:
            results["budget_sensitivity"].append(
                {
                    "budget": budget,
                    "objective": None,
                    "value": None,
                    "cost": None,
                    "n_components": None,
                    "n_sensors": None,
                }
            )
            print(f"Budget ${budget}k: No feasible solution")
            continue

        results["budget_sensitivity"].append(
            {
                "budget": budget,
                "objective": solution["objective"],
                "value": solution["total_value"],
                "cost": solution["total_cost"],
                "n_components": len(solution["selected_components"]),
                "n_sensors": len(solution["selected_sensors"]),
            }
        )
        print(
            f"Budget ${budget}k: Value={solution['total_value']}, "
            f"Cost=${solution['total_cost']}k, "
            f"Components={len(solution['selected_components'])}"
        )

    print("\n=== RESOURCE SENSITIVITY ANALYSIS ===")
    for resources in resource_range:
        candidate_solver = DigitalTwinScopingSolver(
            components=components,
            sensors=sensor_locations,
            coverage_matrix=coverage_matrix,
            budget_limit=budget_limit,
            resource_limit=resources,
            complexity_limit=complexity_limit,
            min_value_required=min_value_required,
            lambda_penalty=lambda_penalty,
        )
        solution, _ = candidate_solver.solve_exhaustive()
        if solution is None:
            results["resource_sensitivity"].append(
                {
                    "resources": resources,
                    "objective": None,
                    "value": None,
                    "cost": None,
                    "n_components": None,
                }
            )
            print(f"Resources {resources} person-hours: No feasible solution")
            continue

        results["resource_sensitivity"].append(
            {
                "resources": resources,
                "objective": solution["objective"],
                "value": solution["total_value"],
                "cost": solution["total_cost"],
                "n_components": len(solution["selected_components"]),
            }
        )
        print(
            f"Resources {resources} person-hours: Value={solution['total_value']}, "
            f"Cost=${solution['total_cost']}k, "
            f"Components={len(solution['selected_components'])}"
        )

    print("\n=== COMPLEXITY-PENALTY SENSITIVITY ANALYSIS ===")
    for lambda_value in lambda_range:
        candidate_solver = DigitalTwinScopingSolver(
            components=components,
            sensors=sensor_locations,
            coverage_matrix=coverage_matrix,
            budget_limit=budget_limit,
            resource_limit=resource_limit,
            complexity_limit=complexity_limit,
            min_value_required=min_value_required,
            lambda_penalty=lambda_value,
        )
        solution, _ = candidate_solver.solve_exhaustive()
        if solution is None:
            results["lambda_sensitivity"].append(
                {
                    "lambda": lambda_value,
                    "objective": None,
                    "value": None,
                    "cost": None,
                    "complexity": None,
                    "n_components": None,
                }
            )
            print(f"Complexity penalty {lambda_value}: No feasible solution")
            continue

        results["lambda_sensitivity"].append(
            {
                "lambda": lambda_value,
                "objective": solution["objective"],
                "value": solution["total_value"],
                "cost": solution["total_cost"],
                "complexity": solution["total_complexity"],
                "n_components": len(solution["selected_components"]),
            }
        )
        print(
            f"Complexity penalty {lambda_value}: Value={solution['total_value']}, "
            f"Complexity={solution['total_complexity']}, "
            f"Components={len(solution['selected_components'])}"
        )

    return results


sensitivity_results = sensitivity_analysis(
    components=components,
    sensor_locations=sensor_locations,
    coverage_matrix=coverage_matrix,
    budget_limit=budget_limit,
    resource_limit=resource_limit,
    complexity_limit=complexity_limit,
    min_value_required=min_value_required,
    lambda_penalty=lambda_penalty,
)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle(
    "Sensitivity Analysis: Terminal Digital Twin Scoping",
    fontsize=16,
    fontweight="bold",
)

# A missing feasible solution is plotted as a gap, not as a numerical zero.
ax1 = axes[0, 0]
budget_data = sensitivity_results["budget_sensitivity"]
budgets = [row["budget"] for row in budget_data]
budget_values = [
    np.nan if row["value"] is None else row["value"] for row in budget_data
]
budget_component_counts = [
    np.nan if row["n_components"] is None else row["n_components"]
    for row in budget_data
]
ax1_twin = ax1.twinx()
line1 = ax1.plot(
    budgets, budget_values, "b-o", linewidth=2, markersize=7, label="Total value"
)
line2 = ax1_twin.plot(
    budgets,
    budget_component_counts,
    "r-s",
    linewidth=2,
    markersize=7,
    label="Components selected",
)
ax1.set_xlabel("Budget limit (thousands of dollars)")
ax1.set_ylabel("Total operational value (score)", color="b")
ax1_twin.set_ylabel("Number of components", color="r")
ax1.set_title("Budget sensitivity")
ax1.grid(True, alpha=0.3)
lines = line1 + line2
ax1.legend(lines, [line.get_label() for line in lines], loc="best")

ax2 = axes[0, 1]
resource_data = sensitivity_results["resource_sensitivity"]
resources = [row["resources"] for row in resource_data]
resource_values = [
    np.nan if row["value"] is None else row["value"] for row in resource_data
]
resource_component_counts = [
    np.nan if row["n_components"] is None else row["n_components"]
    for row in resource_data
]
ax2_twin = ax2.twinx()
line1 = ax2.plot(
    resources,
    resource_values,
    "b-o",
    linewidth=2,
    markersize=7,
    label="Total value",
)
line2 = ax2_twin.plot(
    resources,
    resource_component_counts,
    "r-s",
    linewidth=2,
    markersize=7,
    label="Components selected",
)
ax2.set_xlabel("Resource limit (person-hours)")
ax2.set_ylabel("Total operational value (score)", color="b")
ax2_twin.set_ylabel("Number of components", color="r")
ax2.set_title("Resource sensitivity")
ax2.grid(True, alpha=0.3)
lines = line1 + line2
ax2.legend(lines, [line.get_label() for line in lines], loc="best")

ax3 = axes[1, 0]
lambda_data = sensitivity_results["lambda_sensitivity"]
lambda_values = [row["lambda"] for row in lambda_data]
selected_values = [
    np.nan if row["value"] is None else row["value"] for row in lambda_data
]
selected_complexities = [
    np.nan if row["complexity"] is None else row["complexity"]
    for row in lambda_data
]
ax3_twin = ax3.twinx()
line1 = ax3.plot(
    lambda_values,
    selected_values,
    "b-o",
    linewidth=2,
    markersize=7,
    label="Total value",
)
line2 = ax3_twin.plot(
    lambda_values,
    selected_complexities,
    "g-^",
    linewidth=2,
    markersize=7,
    label="System complexity",
)
ax3.set_xlabel("Complexity-penalty parameter (lambda)")
ax3.set_ylabel("Total operational value (score)", color="b")
ax3_twin.set_ylabel("System complexity (score)", color="g")
ax3.set_title("Complexity-penalty sensitivity")
ax3.grid(True, alpha=0.3)
ax3.set_xscale("log")
lines = line1 + line2
ax3.legend(lines, [line.get_label() for line in lines], loc="best")

# Compare the base case with the endpoint scenarios already evaluated above.
ax4 = axes[1, 1]
scenario_names = [
    "Base case",
    "Highest budget",
    "Highest resource limit",
    "Lowest complexity penalty",
]
scenario_objectives = [
    np.nan if best_solution is None else best_solution["objective"],
    np.nan if budget_data[-1]["objective"] is None else budget_data[-1]["objective"],
    np.nan if resource_data[-1]["objective"] is None else resource_data[-1]["objective"],
    np.nan if lambda_data[0]["objective"] is None else lambda_data[0]["objective"],
]
scenario_colors = ["green", "lightblue", "lightgreen", "gold"]
bars = ax4.bar(
    scenario_names,
    scenario_objectives,
    color=scenario_colors,
    alpha=0.75,
    edgecolor="black",
)
ax4.set_ylabel("Objective function value")
ax4.set_title("Objective values across evaluated scenarios")
ax4.tick_params(axis="x", rotation=30)
ax4.grid(True, alpha=0.3, axis="y")
finite_objectives = [value for value in scenario_objectives if np.isfinite(value)]
label_offset = max(finite_objectives) * 0.01 if finite_objectives else 0.01
for bar, objective in zip(bars, scenario_objectives):
    if np.isfinite(objective):
        ax4.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + label_offset,
            f"{objective:.4f}",
            ha="center",
            va="bottom",
            fontsize=9,
        )

plt.tight_layout()
plt.show()


## Interpreting the method and its results

The model translates a scoping decision into binary component and sensor choices. Coverage creates the essential link between those choices: selecting a component requires at least one sensor that can monitor it. Budget, resources, complexity, and minimum value determine which covered selections are feasible. The objective then ranks feasible selections by value relative to total cost, with an explicit complexity penalty.

Exact exhaustive enumeration guarantees that the best feasible selection is found among the candidates evaluated for this finite instance. Its main limitation is computational growth: the number of subsets increases exponentially with the number of components and sensors. The method is therefore useful for small instances, transparent model checking, and parameter sensitivity analysis.

The sensitivity analysis changes one parameter at a time while keeping the remaining model inputs fixed. A changed solution indicates that the varied parameter affects the best feasible selection under this model; an unchanged solution means only that the tested values did not alter the selected plan. These results are conditional on the stated data and assumptions. They do not establish that the same parameter will be most restrictive in other terminals or larger instances.

### Modeling considerations

- Coverage requirements can make a component subset infeasible even when its component costs and resources satisfy their limits.
- The value-to-cost ratio and complexity penalty express distinct priorities in a single scalar objective; the penalty parameter controls the weight assigned to complexity.
- The model includes no explicit component interdependencies because none are defined in the example data. Such relationships would require additional model inputs and constraints.
- Costs and scores are treated as known values. Uncertainty, operational dynamics, and sensor capacity are not represented.
- For larger instances, the exhaustive search may become impractical. Any extension should preserve the decision meaning and constraints required by the application.